# Feature processing

Prepare an hourly modeling table for next-hour BTC direction prediction, using market features from BTC, ETH, SOL, and XRP alongside FinBERT news sentiment.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

In [2]:
market = pd.read_parquet(DATA / "market_hourly.parquet")
store = pd.read_parquet(DATA / "feature_store.parquet")

for frame in (market, store):
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)

market = market.sort_values("timestamp").reset_index(drop=True)
store = store.sort_values("timestamp").reset_index(drop=True)

## Features and target

Predictors capture returns, volatility, moving averages, funding and positioning, market sessions, and news sentiment. Pair them with the next hourly BTC return to define the target: 1 for an increase in the closing price and 0 otherwise.

In [3]:
FEATURE_COLUMNS = [column for column in store.columns if column != "timestamp"]

features = store.merge(
    market[["timestamp", "btc_close"]].rename(columns={"btc_close": "btc_close_raw"}),
    on="timestamp",
    how="inner",
    validate="one_to_one",
).sort_values("timestamp")

In [4]:
features["next_timestamp"] = features["timestamp"].shift(-1)
features["future_btc_return_1h"] = np.log(
    features["btc_close_raw"].shift(-1) / features["btc_close_raw"]
)

features = features.loc[
    features["next_timestamp"].eq(features["timestamp"] + pd.Timedelta(hours=1))
].copy()

features["target"] = (features["future_btc_return_1h"] > 0).astype(int)

In [5]:
features = (
    features[["timestamp", *FEATURE_COLUMNS, "future_btc_return_1h", "target"]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)

features.to_parquet(DATA / "features.parquet", index=False)

print(f"Modeling table: {len(features):,} hours, {len(FEATURE_COLUMNS)} predictors")

Modeling table: 4,440 hours, 136 predictors
